# Gamatechno AI Support Copilot — Multi-Task Ticket Triage (IndoBERT PEFT/LoRA + Focal Loss + OOF Calibration)

**Target Posisi & Instansi:** AI untuk Customer/Technical Support — PT Gamatechno Indonesia  
**Arsitektur:**
1. **Shared Backbone:** `indobenchmark/indobert-base-p1` (Frozen Base Parameters + **PEFT/LoRA** `r=16, lora_alpha=32, lora_dropout=0.10` pada modul `["query", "key", "value", "dense"]`).
2. **Dual Classification Heads (768 → 256 → 4):**
   - `CategoryHead`: Cross-Entropy Loss dengan `label_smoothing = 0.05` (`Hardware_POS`, `Billing_Payment`, `Academic_Portal`, `Warranty_Claim`).
   - `UrgencyHead`: Multi-Class Focal Loss ($\gamma = 2.0$, bobot terbalik frekuensi kelas $\alpha_t$ dari `train_tickets.csv`) (`Low`, `Medium`, `High`, `Critical`).
3. **Out-of-Fold (OOF) Calibration:** Temperature Scaling $T$ & optimasi ambang batas $\theta_{\text{crit}}^* \in [0.20, 0.60]$ untuk memaksimalkan $F_{\beta=1.5}$ pada kelas `Critical`.

In [ ]:
# 1. Setup Environment & Clone Repository (Otomatis menyesuaikan jika dijalankan di Google Colab)
import os
from pathlib import Path

if not Path("configs/config.yaml").exists():
    if not Path("customerServiceTicketRAG").exists():
        !git clone https://github.com/TheGasz/customerServiceTicketRAG.git
    os.chdir("customerServiceTicketRAG")

!pip install -q peft transformers accelerate lightgbm scikit-learn pandas pyyaml matplotlib seaborn

import torch
print("Working Directory:", Path.cwd())
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Verifikasi Dataset Tiket & Audit Zero Data Leakage (GroupShuffleSplit by user_id)
import pandas as pd
from src.preprocessing.splitter import verify_zero_user_leakage

train_df = pd.read_csv("data/tickets/train_tickets.csv")
val_df = pd.read_csv("data/tickets/val_tickets.csv")
test_df = pd.read_csv("data/tickets/test_tickets.csv")

verify_zero_user_leakage(train_df, val_df, test_df, user_col="user_id")
print(f"Zero Data Leakage Verified! Train={len(train_df)} ({train_df['user_id'].nunique()} users), "
      f"Val={len(val_df)} ({val_df['user_id'].nunique()} users), "
      f"Test={len(test_df)} ({test_df['user_id'].nunique()} users)")

display(pd.DataFrame({
    "Train Category": train_df["category"].value_counts(),
    "Val Category": val_df["category"].value_counts(),
    "Test Category": test_df["category"].value_counts(),
}))

display(pd.DataFrame({
    "Train Urgency": train_df["urgency"].value_counts(),
    "Val Urgency": val_df["urgency"].value_counts(),
    "Test Urgency": test_df["urgency"].value_counts(),
}))

In [ ]:
# 3. Pelatihan & Evaluasi Baseline (TF-IDF + LightGBM di dalam sklearn.pipeline.Pipeline)
from src.triage.baseline_tfidf import train_and_evaluate_baseline

baseline_results = train_and_evaluate_baseline("configs/config.yaml")
baseline_summary = {
    k: v for k, v in baseline_results["test"].items() if not isinstance(v, list)
}
print("=== BASELINE TF-IDF + LIGHTGBM (TEST SPLIT) ===")
display(pd.DataFrame([baseline_summary]))

In [ ]:
# 4. Fine-Tuning Multi-Task IndoBERT + PEFT/LoRA + Focal Loss & OOF Threshold Calibration
import matplotlib.pyplot as plt
from src.triage.train_triage import train_indobert_lora

calib_and_history = train_indobert_lora(config_path="configs/config.yaml")

print("\n=== RINGKASAN PARAMETER & KALIBRASI OOF ===")
print("Parameter Summary:", calib_and_history["parameter_summary"])
print(f"Optimal Temperature (T): {calib_and_history['temperature']}")
print(f"Optimal Critical Threshold (theta_crit*): {calib_and_history['critical_threshold']}")
print(f"Val Critical F_beta=1.5: {calib_and_history['val_critical_fbeta_1_5']} "
      f"(Precision={calib_and_history['val_critical_precision']}, Recall={calib_and_history['val_critical_recall']})")

# Visualisasi kurva pelatihan per Epoch
hist_df = pd.DataFrame(calib_and_history["training_history"])
display(hist_df)

fig, ax1 = plt.subplots(1, 2, figsize=(12, 4))
ax1[0].plot(hist_df["epoch"], hist_df["train_loss"], marker="o", linewidth=2, color="#d95f02")
ax1[0].set_title("Training Multi-Task Loss (CE + Focal Loss)")
ax1[0].set_xlabel("Epoch")
ax1[0].set_ylabel("Loss")
ax1[0].grid(True, alpha=0.3)

ax1[1].plot(hist_df["epoch"], hist_df["val_category_macro_f1"], marker="s", label="Val Category Macro-F1", linewidth=2)
ax1[1].plot(hist_df["epoch"], hist_df["val_urgency_macro_f1"], marker="^", label="Val Urgency Macro-F1", linewidth=2)
ax1[1].set_title("Validation Macro-F1 Progression")
ax1[1].set_xlabel("Epoch")
ax1[1].set_ylabel("Macro-F1")
ax1[1].legend()
ax1[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Evaluasi Komprehensif pada Test Split, Slicing Error Analysis & Confusion Matrix
from IPython.display import Image, display
from src.triage.evaluate_triage import evaluate_all_triage_models

eval_report = evaluate_all_triage_models("configs/config.yaml")

print("=== 1. PERBANDINGAN MODEL PADA HELD-OUT TEST SPLIT ===")
models_df = pd.DataFrame(eval_report["models"]).T
display(models_df)

print(f"\nNER Exact-Match F1: {eval_report['entity_exact_match_f1']} (Target >= 0.95)")
print(f"Formal vs Slang/Typo Gap: {eval_report['formal_vs_slang_gap'] * 100:.2f}% (Target <= 8.0%)")

print("\n=== 2. SLICING ERROR ANALYSIS PER SEGMEN TIKET ===")
slicing_df = pd.DataFrame(eval_report["slicing_analysis"]).T
slicing_df["delta_improvement"] = (
    slicing_df["indobert_lora_macro_f1"] - slicing_df["baseline_macro_f1"]
).round(4)
display(slicing_df)

print("\n=== 3. TOP-10 PREDIKSI SALAH TERBURUK (TOP-LOSS ERRORS) ===")
display(pd.DataFrame(eval_report["top_10_worst_predictions"]))

print("\n=== 4. VISUALISASI CONFUSION MATRIX ===")
display(Image(filename="reports/confusion_matrix_triage.png"))

In [ ]:
# 6. Bungkus Seluruh Artefak Model & Laporan Evaluasi ke ZIP untuk Diunduh
import zipfile
from pathlib import Path

artifact_zip = Path("trained_indobert_lora_artifacts.zip")
paths_to_pack = [
    Path("models/indobert_lora"),
    Path("models/oof_thresholds.json"),
    Path("models/baseline_tfidf_lgbm.joblib"),
    Path("reports/triage_benchmark.json"),
    Path("reports/confusion_matrix_triage.png"),
    Path("reports/triage_slicing_report.md"),
]

with zipfile.ZipFile(artifact_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_pack:
        if item.is_dir():
            for sub in item.rglob("*"):
                if sub.is_file():
                    zf.write(sub, sub.as_posix())
        elif item.is_file():
            zf.write(item, item.as_posix())

print(f"Berhasil membungkus seluruh artefak ke: {artifact_zip.resolve()} ({artifact_zip.stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(str(artifact_zip))
except ImportError:
    print("Berjalan di luar Google Colab; file zip tersimpan di direktori kerja.")